---

# កិច្ចការផ្ទះ 🏠

បំពេញលំហាត់ខាងក្រោម ដើម្បីធ្វើឱ្យជាប់ក្នុងអ្វីដែលអ្នករៀនថ្ងៃនេះ។ ការងារនីមួយៗបន្តលើប្រព័ន្ធ POS ដែលអ្នកបានរៀបចំក្នុងមេរៀននេះ។

---

## កិច្ចការទី 1 — បន្ថែមផលិតផលច្រើនជាងនេះ (ងាយ)

ដោយប្រើ Django shell បន្ថែម **ផលិតផលយ៉ាងហោចណាស់ 4 ទៀត** ក្នុង **ប្រភេទ 2 ផ្សេងគ្នាយ៉ាងហោចណាស់**។ ត្រូវប្រាកដថាផលិតផលនីមួយៗមី barcode ពិសេស។

```python
from sales.models import Product

Product.objects.create(
    name='...',
    category='...',   # food | electronics | clothing | household | other
    price=...,
    stock=...,
    barcode='...',
)
```

---


### Answer

In [ ]:
python manage.py shell

In [ ]:
from sales.models import Product

Product.objects.create(name="កាហ្វេ",category="food",price=2.50,stock=100,barcode="123456789")
Product.objects.create(name="coconut",category="food",price=1.80,stock=200,barcode="133456789")
Product.objects.create(name="tea",category="food",price=0.50,stock=300,barcode="143456789")
Product.objects.create(name="beer",category="food",price=2.50,stock=400,barcode="153456789")

exit()



## កិច្ចការទី 2 — ការអនុវត្ត ORM (ងាយ–មធ្យម)

បើក Django shell ហើយ run queries ខាងក្រោម។ សរសេរលទ្ធផលរបស់អ្នកក្នុងក្រឡាចម្លើយខាងក្រោម។

1. រាប់ចំនួនផលិតផលសរុបក្នុង database។
2. បង្ហាញផលិតផលទាំងអស់ក្នុងប្រភេទ `electronics` តម្រៀបថោកបំផុតមុន។
3. រក product ទាំងអស់ដែល `stock` តិចជាងឬស្មើ 10 (ការព្រមានស្តុកទាប)។
4. បង្ហាញការបញ្ជាទិញ `paid` ទាំងអស់។
5. Get records `OrderItem` ទាំងអស់សម្រាប់ order `#1` ហើយ print subtotal item នីមួយៗ។

---


### Answer

In [ ]:
python manage.py shell

In [ ]:
from sales.models import Product

# 1. រាប់ចំនួនផលិតផលសរុបក្នុង database។
total_products = Product.objects.count()
print(f"Total products in database: {total_products}")

#2. បង្ហាញផលិតផលទាំងអស់ក្នុងប្រភេទ `electronics` តម្រៀបថោកបំផុតមុន។
electronics_products = Product.objects.filter(category='electronics').order_by('price')
print("Electronics products (sorted by price):")
for product in electronics_products:
    print(f"- {product.name}: ${product.price}")

#3. រក product ទាំងអស់ដែល `stock` តិចជាងឬស្មើ 10 (ការព្រមានស្តុកទាប)។
low_stock_products = Product.objects.filter(stock__lte=5)
print("Products with low stock (5 or less):")
for product in low_stock_products:
    print(f"- {product.name}: {product.stock} in stock")

#4. បង្ហាញការបញ្ជាទិញ `paid` ទាំងអស់។
from sales.models import Order
paid_orders = Order.objects.filter(status='paid')
print("Paid orders:")
for order in paid_orders:
    print(f"- Order #{order.pk} by {order.cashier if order.cashier else 'unknown'}: ${order.total:.2f}")

#5. Get records `OrderItem` ទាំងអស់សម្រាប់ order `#1` ហើយ print subtotal item នីមួយៗ។
from sales.models import OrderItem
order_items = OrderItem.objects.filter(order_id=2)
print("Order items for order #1:")
for item in order_items:
    print(f"- {item.product.name}: {item.quantity} x ${item.unit_price} = ${item.subtotal:.2f}")

    


## កិច្ចការទី 3 — Template លម្អិតសម្រាប់ផលិតផល (មធ្យម)

View `product_detail` មានរួចហើយក្នុង `views.py` ប៉ុន្តែ template នៅមិនទាន់មានទេ។

បង្កើត `sales/templates/sales/product_detail.html` ដែលបង្ហាញ:
- ឈ្មោះផលិតផល និង badge ប្រភេទ
- តម្លៃ
- Barcode
- កម្រិតស្តុក (បង្ហាញការព្រមានប្រសិនបើស្តុក < 10)
- តំណ "← ត្រឡប់ទៅបញ្ជីផលិតផល" ដែលទៅ `/sales/products/`

---


### Answer

បង្កើត file `procut_detail.html` នៅក្នុង `sales/template/sales`

In [ ]:
<!-- sales/templates/sales/product_detail.html -->
<!DOCTYPE html>
<html lang="km">
<head>
    <meta charset="UTF-8">
    <title>🛒 POS — {{ product.name }}</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; background: #f0f4f8; }
        h1   { color: #1a202c; }
        .detail-card { background: white; border-radius: 10px; padding: 30px; margin: 20px 0;
                       box-shadow: 0 2px 6px rgba(0,0,0,0.08); }
        .badge { display: inline-block; padding: 5px 12px; border-radius: 12px; font-size: 0.9em;
                 background: #ebf8ff; color: #2b6cb0; font-weight: bold; }
        .price  { font-size: 2em; font-weight: bold; color: #276749; margin: 10px 0; }
        .stock  { font-size: 1.1em; color: #718096; margin: 10px 0; }
        .low    { color: #c53030; font-weight: bold; }
        .info-row { margin: 15px 0; display: flex; gap: 20px; }
        .info-label { font-weight: bold; color: #4a5568; min-width: 150px; }
        .info-value { color: #2d3748; }
        .back-link { display: inline-block; margin: 20px 0; padding: 10px 20px; background: #2b6cb0; color: white;
                     text-decoration: none; border-radius: 5px; transition: background 0.3s; }
        .back-link:hover { background: #1d4f8c; }
        nav a   { margin-right: 16px; color: #2b6cb0; }
    </style>
</head>
<body>
    <h1>🛒 ព័ត៌មានលម្អិតផលិតផល</h1>
    <nav>
        <a href="/sales/products/">ផលិតផល</a>
        <a href="/sales/orders/">ការបញ្ជាទិញ</a>
        <a href="/admin/">Admin</a>
    </nav>

    <div class="detail-card">
        <h2>{{ product.name }}</h2>
        <span class="badge">{{ product.get_category_display }}</span>

        <div class="info-row">
            <div class="info-label">តម្លៃ:</div>
            <div class="info-value">
                <p class="price">${{ product.price }}</p>
            </div>
        </div>

        {% if product.barcode %}
        <div class="info-row">
            <div class="info-label">Barcode:</div>
            <div class="info-value">{{ product.barcode }}</div>
        </div>
        {% endif %}

        <div class="info-row">
            <div class="info-label">ស្តុក:</div>
            <div class="info-value">
                {% if product.stock <= 5 %}
                <p class="low">⚠️ ស្តុកទាប: {{ product.stock }} នៅសល់</p>
                {% else %}
                <p class="stock">{{ product.stock }} ឯកតា</p>
                {% endif %}
            </div>
        </div>

        <div class="info-row">
            <div class="info-label">ស្ថានភាព:</div>
            <div class="info-value">
                {% if product.is_active %}
                <span style="color: #276749; font-weight: bold;">✓ សកម្ម</span>
                {% else %}
                <span style="color: #c53030; font-weight: bold;">✗ អសកម្ម</span>
                {% endif %}
            </div>
        </div>
    </div>

    <a href="/sales/products/" class="back-link">← ត្រលប់ទៅបញ្ជីផលិតផល</a>
</body>
</html>



## កិច្ចការទី 4 — បន្ថែមការបញ្ជាទិញថ្មីតាម Admin (មធ្យម)

1. ចាប់ផ្ដើម server: `python manage.py runserver`
2. Login នៅ `http://127.0.0.1:8000/admin/`
3. បង្កើត **ការបញ្ជាទិញថ្មី** ជាមួយ status `paid` និង **line items យ៉ាងហោចណាស់ 2** ដោយប្រើ inline form។
4. ចូល `http://127.0.0.1:8000/sales/orders/` ហើយផ្ទៀងផ្ទាត់ថា ការបញ្ជាទិញថ្មី និងសរុបរបស់វា បង្ហាញក្នុងបញ្ជី។

---


### Answer : អនុវត្តតាមការណែនាំខាងលើ


## កិច្ចការទី 5 — Deactivate ផលិតផលមួយ (មធ្យម)

នៅ Django shell:
1. Set `is_active = False` លើផលិតផលមួយ ហើយ save វា។
2. ផ្ទៀងផ្ទាត់ថា វាលែងបង្ហាញនៅ `/sales/products/` (ព្រោះ view filter ដោយ `is_active=True`)។
3. សរសេរ code 2 បន្ទាត់ដែលអ្នកប្រើ។

---


### Answer

In [ ]:
python manage.py shell

In [ ]:
from sales.models import Product

# 1. Set `is_active = False` លើផលិតផលមួយ ហើយ save វា។
#product = Product.objects.first()  # យកផលិតផលដំបូង
product = Product.objects.get(pk=1)  # យកផលិតផលតាម primary key
product.is_active = False
product.save()
print(f"Updated product: {product.name} is_active={product.is_active}")

## កិច្ចការទី 6 — ការប្រឡងបន្ថែម ⭐

បន្ថែម model `Discount` ដែលអាចអនុវត្តទៅការបញ្ជាទិញ:

```python
class Discount(models.Model):
    order       = models.OneToOneField(Order, on_delete=models.CASCADE, related_name='discount')
    description = models.CharField(max_length=200)   # ឧ. "ការបញ្ចុះតម្លៃបុគ្គលិក"
    amount      = models.DecimalField(max_digits=6, decimal_places=2)

    def __str__(self):
        return f"{self.description} (—${self.amount}) នៅការបញ្ជាទិញ #{self.order.pk}"
```

បន្ទាប់មក:
1. Run migrations សម្រាប់ model ថ្មី។
2. ចុះឈ្មោះ `Discount` ក្នុង `admin.py`។
3. ធ្វើបច្ចុប្បន្នភាព `Order.total` ដើម្បីដក discount amount ប្រសិនបើមួយមាន:
   ```python
   @property
   def total(self):
       subtotal = sum(item.subtotal for item in self.items.all())
       try:
           return subtotal - self.discount.amount
       except Discount.DoesNotExist:
           return subtotal
   ```
4. តេស្តវានៅ shell ដោយបង្កើត `Discount` សម្រាប់ order ដែលមានស្រាប់ ហើយផ្ទៀងផ្ទាត់ `order.total` ផ្លាស់ប្ដូរ។

---


### Answer

1. update `sales/models.py`

In [ ]:
# sales/models.py

from django.db import models


class Product(models.Model):
    CATEGORY_CHOICES = [
        ('food',        'អាហារ និងភេសជ្ជៈ'),
        ('electronics', 'អេឡិចត្រូនិក'),
        ('clothing',    'សម្លៀកបំពាក់'),
        ('household',   'គ្រឿងសង្ហារឹម'),
        ('other',       'ផ្សេងៗ'),
    ]

    name       = models.CharField(max_length=200)
    category   = models.CharField(max_length=20, choices=CATEGORY_CHOICES)
    price      = models.DecimalField(max_digits=8, decimal_places=2)   # ឧ. 12.99
    stock      = models.PositiveIntegerField(default=0)                # ចំនួនក្នុងស្តុក
    barcode    = models.CharField(max_length=50, unique=True, blank=True)
    is_active  = models.BooleanField(default=True)                     # លាក់ទំនិញឈប់លក់

    def __str__(self):
        return f"{self.name}  —  ${self.price}  (ស្តុក: {self.stock})"

    class Meta:
        ordering = ['name']


class Order(models.Model):
    STATUS_CHOICES = [
        ('open',       'បានបើក'),
        ('paid',       'បានបង់'),
        ('refunded',   'បានសង'),
        ('cancelled',  'បានលុបចោល'),
    ]

    cashier    = models.CharField(max_length=100)           # ឈ្មោះ ឬ ID បុគ្គលិក
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)    # កំណត់ពេលបង្កើតការបញ្ជាទិញ
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        subtotal = sum(item.subtotal for item in self.items.all())
        try:
            return subtotal - self.discount.amount
        except Discount.DoesNotExist:
            return subtotal
             
    def __str__(self):
        return f"ការបញ្ជាទិញ #{self.pk}  [{self.status.upper()}]  —  ${self.total:.2f}"

    class Meta:
        ordering = ['-created_at']


class OrderItem(models.Model):
    order      = models.ForeignKey(Order, on_delete=models.CASCADE, related_name='items')
    product    = models.ForeignKey(Product, on_delete=models.PROTECT)   # PROTECT ការពារការលុបផលិតផលដែលមានការលក់
    quantity   = models.PositiveIntegerField(default=1)
    unit_price = models.DecimalField(max_digits=8, decimal_places=2)    # តម្លៃនៅពេលលក់

    @property
    def subtotal(self):
        return self.unit_price * self.quantity

    def __str__(self):
        return f"{self.quantity} × {self.product.name}  @  ${self.unit_price}"
    
class Discount(models.Model):
    order       = models.OneToOneField(Order, on_delete=models.CASCADE, related_name='discount')
    description = models.CharField(max_length=200)   # ឧ. "ការបញ្ចុះតម្លៃបុគ្គលិក"
    amount      = models.DecimalField(max_digits=6, decimal_places=2)

    def __str__(self):
        return f"{self.description} (—${self.amount}) នៅការបញ្ជាទិញ #{self.order.pk}"

In [ ]:
python manage.py makemigrations sales
python manage.py migrate

2. ចុះឈ្មោះ `Discount` ក្នុង `admin.py`។

In [ ]:
# sales/admin.py

from django.contrib import admin
from .models import Discount, Product, Order, OrderItem


@admin.register(Product)
class ProductAdmin(admin.ModelAdmin):
    list_display  = ['name', 'category', 'price', 'stock', 'is_active']
    list_filter   = ['category', 'is_active']
    search_fields = ['name', 'barcode']
    ordering      = ['name']


class OrderItemInline(admin.TabularInline):
    """បង្ហាញ order items ផ្ទាល់នៅក្នុងទំព័រកែ Order"""
    model  = OrderItem
    extra  = 1    # ចំនួនជួរទទេដែលបង្ហាញសម្រាប់បន្ថែម items ថ្មី
    fields = ['product', 'quantity', 'unit_price']


class DiscountInline(admin.StackedInline):
    """បង្ហាញ discount ផ្ទាល់នៅក្នុងទំព័រកែ Order"""
    model  = Discount
    extra  = 0    # មិនបង្ហាញជួរទទេដែលលាប់លាក់ (OneToOne ទេ ForeignKey)
    fields = ['description', 'amount']

@admin.register(Discount)
class DiscountAdmin(admin.ModelAdmin):
    list_display  = ['description', 'amount', 'order']
    search_fields = ['description', 'order__pk']

@admin.register(Order)
class OrderAdmin(admin.ModelAdmin):
    list_display  = ['pk', 'cashier', 'status', 'created_at']
    list_filter   = ['status']
    search_fields = ['cashier', 'notes']
    ordering      = ['-created_at']
    inlines       = [OrderItemInline, DiscountInline]    # បង្ហាញ items និង discount នៅក្នុងទម្រង់ order



3. ធ្វើបច្ចុប្បន្នភាព `Order.total` ដើម្បីដក discount amount ប្រសិនបើមួយមាន:

In [ ]:
# sales/models.py

from django.db import models


class Product(models.Model):
    CATEGORY_CHOICES = [
        ('food',        'អាហារ និងភេសជ្ជៈ'),
        ('electronics', 'អេឡិចត្រូនិក'),
        ('clothing',    'សម្លៀកបំពាក់'),
        ('household',   'គ្រឿងសង្ហារឹម'),
        ('other',       'ផ្សេងៗ'),
    ]

    name       = models.CharField(max_length=200)
    category   = models.CharField(max_length=20, choices=CATEGORY_CHOICES)
    price      = models.DecimalField(max_digits=8, decimal_places=2)   # ឧ. 12.99
    stock      = models.PositiveIntegerField(default=0)                # ចំនួនក្នុងស្តុក
    barcode    = models.CharField(max_length=50, unique=True, blank=True)
    is_active  = models.BooleanField(default=True)                     # លាក់ទំនិញឈប់លក់

    def __str__(self):
        return f"{self.name}  —  ${self.price}  (ស្តុក: {self.stock})"

    class Meta:
        ordering = ['name']


class Order(models.Model):
    STATUS_CHOICES = [
        ('open',       'បានបើក'),
        ('paid',       'បានបង់'),
        ('refunded',   'បានសង'),
        ('cancelled',  'បានលុបចោល'),
    ]

    cashier    = models.CharField(max_length=100)           # ឈ្មោះ ឬ ID បុគ្គលិក
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)    # កំណត់ពេលបង្កើតការបញ្ជាទិញ
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        subtotal = sum(item.subtotal for item in self.items.all())
        try:
            return subtotal - self.discount.amount
        except Discount.DoesNotExist:
            return subtotal
             
    def __str__(self):
        return f"ការបញ្ជាទិញ #{self.pk}  [{self.status.upper()}]  —  ${self.total:.2f}"

    class Meta:
        ordering = ['-created_at']


class OrderItem(models.Model):
    order      = models.ForeignKey(Order, on_delete=models.CASCADE, related_name='items')
    product    = models.ForeignKey(Product, on_delete=models.PROTECT)   # PROTECT ការពារការលុបផលិតផលដែលមានការលក់
    quantity   = models.PositiveIntegerField(default=1)
    unit_price = models.DecimalField(max_digits=8, decimal_places=2)    # តម្លៃនៅពេលលក់

    @property
    def subtotal(self):
        return self.unit_price * self.quantity

    def __str__(self):
        return f"{self.quantity} × {self.product.name}  @  ${self.unit_price}"
    
class Discount(models.Model):
    order       = models.OneToOneField(Order, on_delete=models.CASCADE, related_name='discount')
    description = models.CharField(max_length=200)   # ឧ. "ការបញ្ចុះតម្លៃបុគ្គលិក"
    amount      = models.DecimalField(max_digits=6, decimal_places=2)

    def __str__(self):
        return f"{self.description} (—${self.amount}) នៅការបញ្ជាទិញ #{self.order.pk}"

4. តេស្តវានៅ shell ដោយបង្កើត `Discount` សម្រាប់ order ដែលមានស្រាប់ ហើយផ្ទៀងផ្ទាត់ `order.total` ផ្លាស់ប្ដូរ។

In [ ]:
python manage.py shell

In [ ]:
#បង្កើត `Discount` សម្រាប់ order ដែលមានស្រាប់ ហើយផ្ទៀងផ្ទាត់ `order.total` ផ្លាស់ប្ដូរ។
from sales.models import Order, Discount
# យក order ដែលមានស្រាប់ (ឧ. order #1)
order = Order.objects.get(pk=2)
# បង្កើត discount 10% លើ total បច្ចុប្បន្ន
current_total = order.total
discount_amount = float(current_total) * 0.10  # 10% discount
discount = Discount.objects.create(order=order, description="ការបញ្ចុះតម្លៃ 10%", amount=discount_amount)
print(f"Created discount: {discount.description} for order #{order.pk} amounting to ${discount.amount:.2f}")
print(f"New total for order #{order.pk}: ${order.total:.2f}")

change `sales/models.py`

In [ ]:

class Order(models.Model):
    STATUS_CHOICES = [
        ('open',       'បានបើក'),
        ('paid',       'បានបង់'),
        ('refunded',   'បានសង'),
        ('cancelled',  'បានលុបចោល'),
    ]

    cashier    = models.CharField(max_length=100)           # ឈ្មោះ ឬ ID បុគ្គលិក
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)    # កំណត់ពេលបង្កើតការបញ្ជាទិញ
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        subtotal = sum(item.subtotal for item in self.items.all())
        try:
            return float(subtotal) - float(self.discount.amount) #make sure to convert to float for accurate calculation
        except Discount.DoesNotExist:
            return float(subtotal) #make sure to convert to float for consistent return type

    def __str__(self):
        return f"ការបញ្ជាទិញ #{self.pk}  [{self.status.upper()}]  —  ${self.total:.2f}"

    class Meta:
        ordering = ['-created_at']

